In [2]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import os

# ---------------------------------------------------------
# 1. Define the Dictionary for Class Names
# ---------------------------------------------------------
classes = { 
    0: 'Speed limit (20km/h)', 1: 'Speed limit (30km/h)', 2: 'Speed limit (50km/h)', 
    3: 'Speed limit (60km/h)', 4: 'Speed limit (70km/h)', 5: 'Speed limit (80km/h)', 
    6: 'End of speed limit (80km/h)', 7: 'Speed limit (100km/h)', 8: 'Speed limit (120km/h)', 
    9: 'No passing', 10: 'No passing veh over 3.5 tons', 11: 'Right-of-way at intersection', 
    12: 'Priority road', 13: 'Yield', 14: 'Stop', 15: 'No vehicles', 
    16: 'Veh > 3.5 tons prohibited', 17: 'No entry', 18: 'General caution', 
    19: 'Dangerous curve left', 20: 'Dangerous curve right', 21: 'Double curve', 
    22: 'Bumpy road', 23: 'Slippery road', 24: 'Road narrows on the right', 
    25: 'Road work', 26: 'Traffic signals', 27: 'Pedestrians', 28: 'Children crossing', 
    29: 'Bicycles crossing', 30: 'Beware of ice/snow', 31: 'Wild animals crossing', 
    32: 'End speed + passing limits', 33: 'Turn right ahead', 34: 'Turn left ahead', 
    35: 'Ahead only', 36: 'Go straight or right', 37: 'Go straight or left', 
    38: 'Keep right', 39: 'Keep left', 40: 'Roundabout mandatory', 41: 'End of no passing', 
    42: 'End no passing veh > 3.5 tons' 
}

In [3]:
# ---------------------------------------------------------
# 2. Recreate the Network Architecture
# ---------------------------------------------------------
class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=43):
        super(TrafficSignCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [4]:
# ---------------------------------------------------------
# 3. Load the Saved Weights
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = TrafficSignCNN(num_classes=43)
model.load_state_dict(torch.load('traffic_sign_classifier.pth', map_location=device))
model.to(device)
model.eval() # Set model to evaluation mode (turns off dropout)

print(f"Model loaded successfully on {device}!")

Model loaded successfully on cuda!


In [5]:
# ---------------------------------------------------------
# 4. Prepare the Image Transform Pipeline
# ---------------------------------------------------------
test_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

In [6]:
# ---------------------------------------------------------
# 5. Function to Predict a Custom Image
# ---------------------------------------------------------
def predict_image(image_path):
    if not os.path.exists(image_path):
        print(f"Error: File not found at {image_path}")
        return
        
    # Open image using PIL and ensure it's RGB
    img = Image.open(image_path).convert('RGB')
    
    # Preprocess and add a batch dimension (unsqueeze) -> Shape becomes [1, 3, 32, 32]
    input_tensor = test_transforms(img).unsqueeze(0).to(device)
    
    # Run prediction
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)
        confidence, predicted_class_id = torch.max(probabilities, 1)
        
    class_id = predicted_class_id.item()
    sign_name = classes[class_id]
    
    print(f"Image: {os.path.basename(image_path)}")
    print(f"Prediction: {sign_name} (Class ID: {class_id})")
    print(f"Confidence: {confidence.item() * 100:.2f}%\n")


In [8]:
# ---------------------------------------------------------
# 6. Test it out!
# ---------------------------------------------------------
# Point this to any image inside your Kaggle 'Test' folder or an image from Google
my_test_image = 'data/Test/00001.png' 
predict_image(my_test_image)

Image: 00001.png
Prediction: Speed limit (30km/h) (Class ID: 1)
Confidence: 97.97%

